# 02 -- Stage 2: DFDC Frame Extraction & Training (Kaggle)

Designed to run as a **Kaggle Notebook** with a DFDC dataset attached via
"Add Input" (no download/upload needed -- Kaggle mounts it read-only
under `/kaggle/input/...`).

**Handles both DFDC packaging styles**:
- Pre-extracted: `.mp4`/`.avi`/`.mov` files + `metadata.json` directly under the input folder.
- **ZIP archives** (e.g. the common "first N parts" reuploads, ~10GB/part):
  each zip is extracted one at a time into scratch space, processed, then
  **deleted** before moving to the next -- this keeps peak disk usage to
  roughly one part's size, regardless of how many parts you attach (attaching
  all ~100GB at once would otherwise exceed Kaggle's working-disk quota).

Labels come from Kaggle's real `metadata.json` format:
`{"filename.mp4": {"label": "REAL"|"FAKE", ...}}`.

**Run order note**: if you're running this *before* Stage 1 (no
`models/image_model.pth` yet), that's fine -- this notebook falls back to
randomly-initialized weights and trains standalone. Stage 3's merge step
reconciles image + video training regardless of which you ran first.

In [ ]:
import sys, os
PROJECT_ROOT = "/content/ForensicAI_Lab"  # adjust if you cloned/uploaded elsewhere
if not os.path.exists(PROJECT_ROOT):
    PROJECT_ROOT = os.getcwd()  # fallback: running locally from the project root
sys.path.insert(0, PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
else:
    print("Not in Colab/Kaggle setup path -- assuming requirements-train.txt is already installed.")

In [ ]:
import torch, cv2, numpy as np, pandas as pd, json, shutil, zipfile
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

from config.loader import get_settings, resolve_dataset_root, resolve_path
from services.video.extractor import extract_frames, get_video_fps, get_frame_count
from services.video.sampler import sample_frame_indices
from services.preprocessing.pipeline import preprocess_image
from services.detection.model import ForensicMultiHeadNet

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

## Locate the attached dataset

Change `KAGGLE_INPUT_SLUG` to match whatever "Add Input" shows in your
notebook's sidebar. Falls back to `dataset_root/dfdc_subset` if not
running on Kaggle (e.g. you manually placed files there instead).

In [ ]:
KAGGLE_INPUT_SLUG = "dfdc-10"  # <-- change to your attached dataset's folder name
kaggle_input_path = Path("/kaggle/input") / KAGGLE_INPUT_SLUG

zip_source_dir = kaggle_input_path if kaggle_input_path.exists() else dataset_root / "dfdc_subset"
print("Reading DFDC data from:", zip_source_dir)

zip_files = sorted(zip_source_dir.glob("**/*.zip"))
if zip_files:
    print(f"Found {len(zip_files)} zip part(s):")
    for z in zip_files:
        print(f"  {z.name}  ({z.stat().st_size / 1e9:.1f} GB)")
else:
    print("No .zip files found -- assuming the dataset is already extracted at this path.")

## Extract one part at a time, process, then delete it

`MAX_PARTS` and `MAX_VIDEOS_PER_PART` let you do a cheap first test run
before committing to the full ~100GB set -- e.g. set `MAX_PARTS = 1` and
`MAX_VIDEOS_PER_PART = 50` to validate the pipeline end-to-end in minutes,
then raise both for the real run.

In [ ]:
MAX_PARTS = len(zip_files) if zip_files else 1   # lower this for a quick test run, e.g. 1
MAX_VIDEOS_PER_PART = None                        # e.g. 50 for a quick test run; None = no cap
FRAMES_PER_VIDEO = 10

scratch_dir = Path("/kaggle/working/dfdc_scratch") if kaggle_input_path.exists() else Path("/tmp/dfdc_scratch")
frames_out_dir = resolve_path("data/synthetic_data")
frames_out_dir.mkdir(parents=True, exist_ok=True)
frame_manifest_path = resolve_path("data/dfdc_frame_manifest.csv")
if frame_manifest_path.exists():
    frame_manifest_path.unlink()  # start fresh each run; re-run-safe

# Each entry is (path_to_process, needs_zip_extraction)
sources = [(z, True) for z in zip_files] if zip_files else [(zip_source_dir, False)]

total_rows_written = 0

for part_idx, (source, needs_extract) in enumerate(sources[:MAX_PARTS]):
    print(f"\n=== Part {part_idx + 1}/{min(MAX_PARTS, len(sources))}: {source.name} ===")

    if needs_extract:
        if scratch_dir.exists():
            shutil.rmtree(scratch_dir)
        scratch_dir.mkdir(parents=True)
        with zipfile.ZipFile(source) as zf:
            zf.extractall(scratch_dir)
        search_root = scratch_dir
    else:
        search_root = source

    metadata_files = list(search_root.glob("**/metadata.json"))
    if not metadata_files:
        print(f"  No metadata.json found under {search_root}, skipping this part")
        if needs_extract:
            shutil.rmtree(scratch_dir)
        continue

    part_rows = []
    for meta_path in metadata_files:
        video_folder = meta_path.parent
        with open(meta_path) as f:
            metadata = json.load(f)

        items = list(metadata.items())
        if MAX_VIDEOS_PER_PART is not None:
            items = items[:MAX_VIDEOS_PER_PART]

        for filename, info in items:
            video_path = video_folder / filename
            if not video_path.exists():
                continue

            label = "real" if info["label"].upper() == "REAL" else "fake"
            manip_type = "none" if label == "real" else "gan"  # deepfake/face-swap -> closest existing class

            fps = get_video_fps(str(video_path))
            total = get_frame_count(str(video_path))
            indices = set(sample_frame_indices(total, fps, settings.video.sample_fps, max_frames=FRAMES_PER_VIDEO))

            for idx, frame in extract_frames(str(video_path)):
                if idx not in indices:
                    continue
                frame_name = f"{source.stem}_{video_path.stem}_frame{idx}.jpg"
                cv2.imwrite(str(frames_out_dir / frame_name), frame)
                part_rows.append({"filename": str(frames_out_dir / frame_name), "label": label, "type": manip_type})

    pd.DataFrame(part_rows, columns=["filename", "label", "type"]).to_csv(
        frame_manifest_path, mode="a", header=(total_rows_written == 0), index=False
    )
    total_rows_written += len(part_rows)
    print(f"  Extracted {len(part_rows)} frame rows from this part (running total: {total_rows_written})")

    if needs_extract:
        shutil.rmtree(scratch_dir)  # free disk before the next part

print(f"\nDone. {total_rows_written} total frame rows written to {frame_manifest_path}")

## Train on the extracted frames

In [ ]:
class FrameDataset(Dataset):
    """Like Stage 1's image dataset, but reads full paths straight from the
    frame manifest (no images_dir join) and has no ground-truth masks --
    mask loss is zero-weighted for every sample via has_mask=0."""

    def __init__(self, manifest_csv, image_size):
        self.df = pd.read_csv(manifest_csv)
        self.image_size = image_size
        self.label_to_idx = {l: i for i, l in enumerate(settings.labels.binary)}
        manip_classes = settings.labels.manipulation_types
        self.manip_to_idx = {m: i for i, m in enumerate(manip_classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_bgr = cv2.imread(row["filename"])
        x = preprocess_image(image_bgr, tuple(self.image_size))

        label_idx = self.label_to_idx[row["label"]]
        manip_idx = self.manip_to_idx.get(row["type"], 0)
        mask = np.zeros((1, *self.image_size), dtype=np.float32)

        return (
            torch.from_numpy(x),
            torch.tensor(label_idx, dtype=torch.long),
            torch.tensor(manip_idx, dtype=torch.long),
            torch.from_numpy(mask),
            torch.tensor(0.0, dtype=torch.float32),
        )

In [ ]:
frame_dataset = FrameDataset(frame_manifest_path, settings.inference.image_size)
loader = DataLoader(frame_dataset, batch_size=16, shuffle=True, num_workers=2)
print("Frame dataset size:", len(frame_dataset))

In [ ]:
model = ForensicMultiHeadNet(
    backbone=settings.models.backbone,
    in_chans=settings.inference.input_channels,
    num_manip_classes=len(settings.labels.manipulation_types),
).to(device)

image_ckpt = resolve_path(settings.models.image_model_checkpoint)
if image_ckpt.exists():
    model.load_state_dict(torch.load(image_ckpt, map_location=device))
    print("Warm-started from Stage 1 checkpoint:", image_ckpt)
else:
    print("No Stage 1 checkpoint found -- training from randomly-initialized weights. "
          "That's fine; Stage 3's merge step reconciles image + video training either order.")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
cls_loss_fn = torch.nn.CrossEntropyLoss()
manip_loss_fn = torch.nn.CrossEntropyLoss()
mask_loss_fn = torch.nn.BCEWithLogitsLoss(reduction="none")

NUM_EPOCHS = 5  # adjust based on dataset size / Kaggle session limits

In [ ]:
model.train()
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    for x, label_idx, manip_idx, mask, has_mask in loader:
        x, label_idx, manip_idx = x.to(device), label_idx.to(device), manip_idx.to(device)
        mask, has_mask = mask.to(device), has_mask.to(device)

        optimizer.zero_grad()
        out = model(x)

        loss_cls = cls_loss_fn(out["cls_logits"], label_idx)
        loss_manip = manip_loss_fn(out["manip_logits"], manip_idx)
        mask_loss_per_px = mask_loss_fn(out["mask_logits"], mask)
        loss_mask = (mask_loss_per_px.mean(dim=[1, 2, 3]) * has_mask).sum() / (has_mask.sum() + 1e-8)

        loss = loss_cls + loss_manip + loss_mask
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} -- loss: {running_loss / len(loader):.4f}")

In [ ]:
checkpoint_path = resolve_path(settings.models.video_model_checkpoint)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), checkpoint_path)
print("Saved checkpoint to", checkpoint_path)

## Getting the checkpoint out of Kaggle

Kaggle keeps this session's output files under the notebook's "Output"
tab, but that's separate from your Google Drive. To use this checkpoint in
the Colab notebooks (Stage 3 merge, Stage 6 export), download
`models/video_model.pth` from the Output tab and upload it to
`ForensicAI_Lab/models/video_model.pth` on your Drive.